### Agrupamento de `uf` (-> `regiao`) e `cnae_fiscal_principal` (-> `grupo_cnae`)

**Entrada:** `data/dataset_modelagem_final.parquet` (45.876 linhas, 45 colunas) -- já com o artefato EIRELI excluído (notebook `02`) e `natureza_juridica_agrupada` criada (notebook `03`).

**Saída:** o mesmo arquivo, sobrescrito -- linhas mantidas, duas colunas novas (`regiao`, `grupo_cnae`).

#### 1. Motivo do agrupamento

`uf` (27 categorias) e `cnae_fiscal_principal` (7 dígitos, dezenas de subclasses) estourariam o orçamento de ~20 parâmetros do modelo como dummies. Reduzimos a cardinalidade mapeando `uf` para as 5 regiões do IBGE e `cnae_fiscal_principal` para os 3 primeiros dígitos do código (grupo CNAE).

#### 2. Carregamento

Carrega o checkpoint do notebook 03.

In [1]:
import pandas as pd

# checkpoint do notebook 03, já com natureza_juridica_agrupada
CAMINHO = "data/dataset_modelagem_final.parquet"

df = pd.read_parquet(CAMINHO)
print("Carregado:", df.shape)

Carregado: (45876, 45)


#### 3.1 Investigação: `uf` tem 28 valores únicos, mas só existem 27 UFs válidas

Lista os valores únicos de `uf` para achar o 28º.

In [2]:
# lista os valores unicos de uf para identificar o 28o valor (esperado: 27 UFs reais)
valores_uf = sorted(df["uf"].astype(str).unique().tolist())
print(f"Valores unicos de uf ({len(valores_uf)}):", valores_uf)
print()
print("Contagem do valor suspeito 'EX':", (df["uf"] == "EX").sum())

Valores unicos de uf (28): ['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'EX', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR', 'RS', 'SC', 'SE', 'SP', 'TO']



Contagem do valor suspeito 'EX': 2


In [3]:
# 'EX' -- nao e uma UF brasileira. Confirma que sao estabelecimentos
# domiciliados no exterior (mesmo fenomeno de natureza_juridica=2216,
# "Empresa Domiciliada no Exterior", ja visto no notebook 03).
cols = ["uf", "natureza_juridica", "pais", "nome_cidade_exterior", "municipio", "alvo_baixada"]
print(df[df["uf"] == "EX"][cols].to_string())

      uf natureza_juridica pais nome_cidade_exterior municipio  alvo_baixada
2573  EX              2216  628               LONDON      9707             0
8906  EX              2216  249                LEWES      9707             0


#### 3.2 Achado: `'EX'` são empresas domiciliadas no exterior

`'EX'` não é uma UF real -- são as 2 empresas com `natureza_juridica=2216`, já vistas no notebook 03. Como não existe região brasileira válida para elas, o mapeamento abaixo deixa `regiao` nula para essas 2 linhas; a decisão fica para a seção 9.

#### 4. Construção de `regiao` e `grupo_cnae`

Aplica os dois mapeamentos definidos na motivação (seção 1).

In [4]:
# mapeamento padrao do IBGE: 27 UFs -> 5 regioes
MAPA_REGIAO = {
    "AC": "Norte", "AP": "Norte", "AM": "Norte", "PA": "Norte", "RO": "Norte", "RR": "Norte", "TO": "Norte",
    "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste", "MA": "Nordeste", "PB": "Nordeste",
    "PE": "Nordeste", "PI": "Nordeste", "RN": "Nordeste", "SE": "Nordeste",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MS": "Centro-Oeste", "MT": "Centro-Oeste",
    "ES": "Sudeste", "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
    "PR": "Sul", "RS": "Sul", "SC": "Sul",
}

df["regiao"] = df["uf"].map(MAPA_REGIAO)

# 'EX' fica de fora do dicionario de proposito -- nao existe regiao brasileira valida para ela (ver secao 3)
nao_mapeados = df[df["regiao"].isna()]
print("Linhas com 'regiao' nao mapeada:", len(nao_mapeados))
print("UF(s) sem mapeamento:", nao_mapeados["uf"].unique().tolist())

Linhas com 'regiao' nao mapeada: 2
UF(s) sem mapeamento: ['EX']


In [5]:
# primeiros 3 digitos do CNAE = o grupo dentro da divisao 47 (ex.: 4781400 -> 478), reduzindo dezenas
# de subclasses de 7 digitos a um punhado de grupos
df["grupo_cnae"] = df["cnae_fiscal_principal"].str[:3]

print("Valores unicos de grupo_cnae (" + str(df["grupo_cnae"].nunique()) + "):", sorted(df["grupo_cnae"].unique().tolist()))

Valores unicos de grupo_cnae (8): ['471', '472', '473', '474', '475', '476', '477', '478']


#### 5. Crosstabs de confirmação

Usa `dropna=False` para não esconder a categoria de `regiao` não mapeada (as 2 linhas de `uf='EX'`).

In [6]:
# n e taxa de fracasso por regiao, com dropna=False para nao esconder a categoria nao mapeada (uf='EX')
n_regiao = pd.crosstab(df["regiao"], df["alvo_baixada"], dropna=False)
n_regiao.columns = ["n_sobreviveu_0", "n_fracassou_1"]
n_regiao["n_total"] = n_regiao["n_sobreviveu_0"] + n_regiao["n_fracassou_1"]
n_regiao["taxa_fracasso_pct"] = (n_regiao["n_fracassou_1"] / n_regiao["n_total"] * 100).round(1)
n_regiao = n_regiao.sort_values("n_total", ascending=False)
print("=== regiao x alvo_baixada ===")
print(n_regiao.to_string())

=== regiao x alvo_baixada ===
              n_sobreviveu_0  n_fracassou_1  n_total  taxa_fracasso_pct
regiao                                                                 
Sudeste                 8522          12473    20995               59.4
Nordeste                4324           6043    10367               58.3
Sul                     3067           4497     7564               59.5
Centro-Oeste            1637           2501     4138               60.4
Norte                   1246           1564     2810               55.7
NaN                        2              0        2                0.0


In [7]:
# n e taxa de fracasso por grupo_cnae, mesma logica da checagem de regiao acima
n_cnae = pd.crosstab(df["grupo_cnae"], df["alvo_baixada"], dropna=False)
n_cnae.columns = ["n_sobreviveu_0", "n_fracassou_1"]
n_cnae["n_total"] = n_cnae["n_sobreviveu_0"] + n_cnae["n_fracassou_1"]
n_cnae["taxa_fracasso_pct"] = (n_cnae["n_fracassou_1"] / n_cnae["n_total"] * 100).round(1)
n_cnae = n_cnae.sort_values("n_total", ascending=False)
print("=== grupo_cnae x alvo_baixada ===")
print(n_cnae.to_string())

=== grupo_cnae x alvo_baixada ===
            n_sobreviveu_0  n_fracassou_1  n_total  taxa_fracasso_pct
grupo_cnae                                                           
478                   6666          11191    17857               62.7
472                   3086           5237     8323               62.9
475                   2667           3489     6156               56.7
471                   2092           2592     4684               55.3
477                   1755           2308     4063               56.8
474                   1665           1302     2967               43.9
476                    679            869     1548               56.1
473                    188             90      278               32.4


#### 6. Diagnóstico de categorias problemáticas

As 5 regiões reais e os 8 grupos de CNAE têm `n` e taxa de fracasso saudáveis. Só a categoria `regiao` nula (as 2 linhas de `uf='EX'`) é degenerada, como esperado para uma UF que não existe -- nenhuma decisão é tomada aqui, só o diagnóstico.

#### 7. Salvamento intermediário

Salva o dataset com `regiao` e `grupo_cnae`, deixando a decisão sobre as 2 linhas de `regiao` nula para a seção 9.

In [8]:
# checkpoint intermediario: regiao ainda tem 2 linhas nulas (uf='EX'), decisao fica para a secao 9
assert len(df) == 45876, f"Esperado 45876, obtido {len(df)}"

df.to_parquet(CAMINHO, index=False)
print(f"Salvo (sobrescrito) em {CAMINHO}")
print("Shape final:", df.shape)

Salvo (sobrescrito) em data/dataset_modelagem_final.parquet
Shape final: (45876, 47)


#### 8. Resumo

Dataset intermediário: 45.876 linhas x 47 colunas (+2: `regiao`, `grupo_cnae`; `uf` e `cnae_fiscal_principal` mantidas). Pendência em aberto: as 2 linhas com `uf='EX'` (`regiao` nula).

#### 9. Decisão sobre `uf='EX'`: excluir as 2 linhas

Removemos as 2 linhas com `uf='EX'` (empresas no exterior, sem UF real) -- diferente do agrupamento em `natureza_juridica_agrupada`, aqui a variável geográfica simplesmente não se aplica ao caso. Volume irrisório (2 em 45.876), sem efeito prático na estimação. O racional completo está no comentário da célula de código abaixo.

In [9]:
# Remove as 2 linhas com uf='EX' (empresas domiciliadas no exterior -- Londres e Lewes, as mesmas ja
# identificadas como natureza_juridica=2216 no notebook 03). Elas nao tem UF real, entao qualquer valor
# atribuido a regiao seria artificial; diferente do agrupamento em natureza_juridica_agrupada (onde a
# observacao era valida e so o rotulo era raro), aqui a variavel geografica nao se aplica ao caso. Volume
# irrisorio (2 em 45.876, ~0.004%) -- exclusao sem efeito pratico na estimacao.
df_final = df[df["uf"] != "EX"].copy()
print("Linhas removidas (uf=='EX'):", len(df) - len(df_final))
print("Shape apos exclusao:", df_final.shape)

# trava de sanidade: confirma a contagem exata de linhas removidas
assert len(df_final) == 45874, f"Esperado 45874, obtido {len(df_final)}"
print("Assert OK: 45874 linhas")

# trava de sanidade: garante que a exclusao eliminou toda regiao nula, nao so reduziu
assert df_final["regiao"].isna().sum() == 0, "Ainda sobra regiao nula!"
print("Assert OK: nenhuma linha com 'regiao' nula")

df_final.to_parquet(CAMINHO, index=False)
print(f"Salvo (sobrescrito) em {CAMINHO}")
print("Shape final:", df_final.shape)

Linhas removidas (uf=='EX'): 2
Shape apos exclusao: (45874, 47)
Assert OK: 45874 linhas
Assert OK: nenhuma linha com 'regiao' nula


Salvo (sobrescrito) em data/dataset_modelagem_final.parquet
Shape final: (45874, 47)
